# Lab 4 - Silver Curated Table and SCD Type 1

MERGEs the prepared Silver updates into the main Silver table. This table keeps only the current state per `show_id`, so it implements SCD Type 1.


In [0]:
%run ./lab4_00_config


In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

prepared_updates = spark.table(silver_updates_table)


In [0]:
if not spark.catalog.tableExists(silver_curated_table):
    (
        prepared_updates.limit(0)
        .withColumn("silver_created_at", F.current_timestamp())
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(silver_curated_table)
    )

target = DeltaTable.forName(spark, silver_curated_table)
source_for_merge = prepared_updates.withColumn("silver_created_at", F.current_timestamp())

update_map = {
    col_name: f"source.{col_name}"
    for col_name in source_for_merge.columns
    if col_name != "silver_created_at"
}

insert_map = {col_name: f"source.{col_name}" for col_name in source_for_merge.columns}

(
    target.alias("target")
    .merge(source_for_merge.alias("source"), "target.show_id = source.show_id")
    .withSchemaEvolution()
    .whenMatchedUpdate(condition="target._record_hash <> source._record_hash", set=update_map)
    .whenNotMatchedInsert(values=insert_map)
    .execute()
)


In [0]:
display(spark.table(silver_curated_table).orderBy("show_id").limit(20))

duplicate_check = (
    spark.table(silver_curated_table)
    .groupBy("show_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate show_id rows:", duplicate_check.count())
display(duplicate_check)
